# Speed-band visuals (project 1.3)

Draws maps and charts from whatever the collector has saved so far. Run it any time; it only reads `data_raw/`, never changes it.

- Python kernel, run from the project folder. Needs `polars` and `matplotlib` (`pip install polars matplotlib`).
- Run cell 1 first, then any of the plot cells. Change the settings at the top of each cell.
- Figures are saved to `figures/`.

Data: LTA DataMall Traffic Speed Bands, Singapore Open Data Licence v1.0. AI disclosure: drafted with Claude (Anthropic), 14 Oct 2026; log in `AI_USE.md`.

## 1. Setup and helpers

In [ ]:
import gzip, json
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from matplotlib.collections import LineCollection
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

PROJ = Path.cwd()
RAW = PROJ / "data_raw"
FIG = PROJ / "figures"
FIG.mkdir(exist_ok=True)

BAND_LABELS = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70+"]   # km/h, bands 1-8
BAND_COLOURS = ["#d73027", "#f46d43", "#fdae61", "#fee08b", "#d9ef8b", "#a6d96a", "#66bd63", "#1a9850"]
NO_DATA = "#bdbdbd"
CATEGORY_NAMES = {1: "Expressways", 2: "Major arterial roads", 3: "Arterial roads", 4: "Minor arterial roads",
                  5: "Small roads", 6: "Slip roads", 8: "Short tunnels"}           # LTA API guide v6.10, 2.19
CAPTION = "Contains information from LTA DataMall (Traffic Speed Bands), Singapore Open Data Licence v1.0"


def load_links() -> pl.DataFrame:
    """Link attributes and coordinates from the latest full raw snapshot (saved hourly)."""
    files = sorted((RAW / "speedbands_raw").rglob("*.jsonl.gz"))
    if not files:
        raise FileNotFoundError("No raw snapshot yet: the collector saves one per hour.")
    records = []
    with gzip.open(files[-1], "rt", encoding="utf-8") as f:
        for line in f:
            records.extend(json.loads(json.loads(line)["body"])["value"])
    return (pl.DataFrame(records, infer_schema_length=None)
              .select(pl.col("LinkID").cast(pl.Utf8), "RoadName",
                      pl.col("RoadCategory").cast(pl.Int64),
                      *[pl.col(c).cast(pl.Float64) for c in ["StartLon", "StartLat", "EndLon", "EndLat"]]))


def list_snapshots() -> pl.DataFrame:
    """All compact snapshots with their retrieval time (from the file name)."""
    files = sorted((RAW / "speedbands").rglob("sb_*.csv.gz"))
    return pl.DataFrame({"path": [str(f) for f in files],
                         "t": [datetime.strptime(f.stem.replace(".csv", "")[3:], "%Y%m%d_%H%M%S") for f in files]})


def load_snapshot(path: str) -> pl.DataFrame:
    with gzip.open(path, "rb") as f:
        df = pl.read_csv(f.read(), schema_overrides={"LinkID": pl.Utf8, "SpeedBand": pl.Int64})
    return df.select("LinkID", "SpeedBand", "last_updated")


def pick_snapshot(when: str = "latest") -> dict:
    """'latest' or 'YYYY-MM-DD HH:MM' (Singapore time): returns the nearest saved snapshot."""
    snaps = list_snapshots()
    if snaps.height == 0:
        raise FileNotFoundError("No snapshots saved yet.")
    if when == "latest":
        return snaps.row(-1, named=True)
    target = datetime.strptime(when, "%Y-%m-%d %H:%M")
    return snaps.with_columns((pl.col("t") - target).abs().alias("d")).sort("d").row(0, named=True)


def draw_map(ax, df: pl.DataFrame, title: str) -> None:
    """df needs StartLon, StartLat, EndLon, EndLat, SpeedBand (0 = no data)."""
    segs = np.stack([df.select("StartLon", "StartLat").to_numpy(), df.select("EndLon", "EndLat").to_numpy()], axis=1)
    bands = df["SpeedBand"].to_numpy()
    colours = [BAND_COLOURS[b - 1] if 1 <= b <= 8 else NO_DATA for b in bands]
    ax.add_collection(LineCollection(segs, colors=colours, linewidths=1.2, capstyle="round"))
    ax.autoscale()
    ax.set_aspect(1 / np.cos(np.deg2rad(1.35)))   # correct lon/lat scale at Singapore's latitude
    ax.set_axis_off()
    ax.set_title(title, fontsize=11, loc="left")


def band_legend(fig) -> None:
    handles = [Patch(color=c, label=l) for c, l in zip(BAND_COLOURS, BAND_LABELS)] + [Patch(color=NO_DATA, label="no data")]
    fig.legend(handles=handles, title="Speed (km/h)", loc="center right", frameon=False)


links = load_links()
snaps = list_snapshots()
print(f"Links in latest raw snapshot: {links.height:,}")
print(f"Compact snapshots saved: {snaps.height:,}, from {snaps['t'].min()} to {snaps['t'].max()}")
print(snaps.group_by(pl.col("t").dt.date().alias("date")).len().sort("date"))

## 2. Map of one moment
Set `WHEN` to `"latest"` or a time like `"2026-10-15 08:30"`, and `CATEGORY` (1 = expressways, 2 = major arterials).

In [ ]:
WHEN = "latest"
CATEGORY = 1

snap = pick_snapshot(WHEN)
df = links.filter(pl.col("RoadCategory") == CATEGORY).join(load_snapshot(snap["path"]), on="LinkID", how="inner")

fig, ax = plt.subplots(figsize=(11, 6.5))
draw_map(ax, df, f"{CATEGORY_NAMES[CATEGORY]}: speed bands at {snap['t']:%a %d %b %Y, %H:%M} ({df.height:,} links)")
band_legend(fig)
fig.text(0.01, 0.01, CAPTION, fontsize=7, color="grey")
fig.subplots_adjust(right=0.85)
out = FIG / f"map_cat{CATEGORY}_{snap['t']:%Y%m%d_%H%M}.png"
fig.savefig(out, dpi=200, facecolor="white")
plt.show()
print("Saved", out)

## 3. Compare two moments side by side
For example, night (free flow) vs the morning peak.

In [ ]:
WHEN_A, WHEN_B = "2026-10-15 03:00", "2026-10-15 08:30"
CATEGORY = 1

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, when in zip(axes, [WHEN_A, WHEN_B]):
    snap = pick_snapshot(when)
    df = links.filter(pl.col("RoadCategory") == CATEGORY).join(load_snapshot(snap["path"]), on="LinkID", how="inner")
    draw_map(ax, df, f"{snap['t']:%a %d %b, %H:%M}")
band_legend(fig)
fig.suptitle(f"{CATEGORY_NAMES[CATEGORY]}: speed bands", x=0.01, ha="left")
fig.text(0.01, 0.01, CAPTION, fontsize=7, color="grey")
fig.subplots_adjust(right=0.9)
out = FIG / f"map_compare_cat{CATEGORY}.png"
fig.savefig(out, dpi=200, facecolor="white")
plt.show()
print("Saved", out)

## 4. Time–space diagram for one road on one day
Each row is a link, ordered along the road; each column is a snapshot. Recurring peaks appear as blocks at the same time every weekday; incidents appear as one-off blocks. The two directions are plotted separately.

Reading a full day takes a minute or two (each snapshot covers the whole network).

In [ ]:
ROAD = "PAN ISLAND EXPRESSWAY"   # exact RoadName, upper case
DATE = "latest"                   # or "2026-10-15"

day = snaps["t"].max().date() if DATE == "latest" else datetime.strptime(DATE, "%Y-%m-%d").date()
road = links.filter(pl.col("RoadName") == ROAD)
if road.height == 0:
    raise ValueError(f"No links named {ROAD!r}. Some names: {links.filter(pl.col('RoadCategory') == 1)['RoadName'].unique().head(10).to_list()}")

# Order links along the road: project midpoints on the road's main axis; split directions by heading.
mid = np.column_stack([(road["StartLon"] + road["EndLon"]) / 2, (road["StartLat"] + road["EndLat"]) / 2])
axis = np.linalg.svd(mid - mid.mean(axis=0))[2][0]
heading = np.column_stack([road["EndLon"] - road["StartLon"], road["EndLat"] - road["StartLat"]]) @ axis
road = road.with_columns(pl.Series("pos", (mid - mid.mean(axis=0)) @ axis), pl.Series("dir", np.where(heading >= 0, "A", "B")))

day_snaps = snaps.filter(pl.col("t").dt.date() == day)
ids = road["LinkID"]
frames = [load_snapshot(p).filter(pl.col("LinkID").is_in(ids)).with_columns(pl.lit(t).alias("t"))
          for p, t in zip(day_snaps["path"], day_snaps["t"])]
panel = pl.concat(frames).join(road.select("LinkID", "pos", "dir"), on="LinkID")
print(f"{ROAD} on {day}: {road.height} links, {day_snaps.height} snapshots")

cmap = ListedColormap([NO_DATA] + BAND_COLOURS)
norm = BoundaryNorm(np.arange(-0.5, 9.5), cmap.N)
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for ax, d in zip(axes, ["A", "B"]):
    sub = panel.filter(pl.col("dir") == d)
    if sub.height == 0:
        ax.set_visible(False)
        continue
    order = sub.select("LinkID", "pos").unique().sort("pos")["LinkID"].to_list()
    wide = sub.pivot(on="LinkID", index="t", values="SpeedBand").sort("t")
    times = wide["t"].to_list()
    grid = wide.select(order).to_numpy().T.astype(float)
    grid = np.where(np.isnan(grid), 0, grid)
    hours = [(t - datetime.combine(day, datetime.min.time())).total_seconds() / 3600 for t in times]
    ax.imshow(grid, aspect="auto", cmap=cmap, norm=norm, interpolation="nearest", origin="lower",
              extent=[hours[0], hours[-1] + 5 / 60, 0, len(order)])
    ax.set_ylabel(f"Direction {d}\nposition along road")
    ax.set_yticks([])
axes[-1].set_xlabel("Hour of day (SGT)")
axes[-1].set_xticks(range(0, 25, 2))
fig.suptitle(f"{ROAD}: speed bands on {day:%a %d %b %Y}", x=0.01, ha="left")
band_legend(fig)
fig.text(0.01, 0.01, CAPTION, fontsize=7, color="grey")
fig.subplots_adjust(right=0.88)
out = FIG / f"timespace_{ROAD.replace(' ', '_').lower()}_{day:%Y%m%d}.png"
fig.savefig(out, dpi=200, facecolor="white")
plt.show()
print("Saved", out)

## 5. Share of links in each band over one day
A quick view of how the whole road category slows down at peaks.

In [ ]:
CATEGORY = 1
DATE = "latest"

day = snaps["t"].max().date() if DATE == "latest" else datetime.strptime(DATE, "%Y-%m-%d").date()
ids = links.filter(pl.col("RoadCategory") == CATEGORY)["LinkID"]
day_snaps = snaps.filter(pl.col("t").dt.date() == day)
share = (pl.concat([load_snapshot(p).filter(pl.col("LinkID").is_in(ids)).group_by("SpeedBand").len()
                      .with_columns(pl.lit(t).alias("t")) for p, t in zip(day_snaps["path"], day_snaps["t"])])
           .pivot(on="SpeedBand", index="t", values="len").sort("t").fill_null(0))
bands_present = [str(b) for b in range(0, 9) if str(b) in share.columns]
totals = share.select(bands_present).sum_horizontal().to_numpy()
hours = [(t - datetime.combine(day, datetime.min.time())).total_seconds() / 3600 for t in share["t"]]
stack = [share[b].to_numpy() / totals for b in bands_present]
colours = [NO_DATA if b == "0" else BAND_COLOURS[int(b) - 1] for b in bands_present]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.stackplot(hours, stack, colors=colours, linewidth=0)
ax.set_xlim(0, 24); ax.set_ylim(0, 1)
ax.set_xticks(range(0, 25, 2)); ax.set_xlabel("Hour of day (SGT)"); ax.set_ylabel("Share of links")
ax.set_title(f"{CATEGORY_NAMES[CATEGORY]}: share of links by speed band, {day:%a %d %b %Y}", loc="left", fontsize=11)
band_legend(fig)
fig.text(0.01, 0.01, CAPTION, fontsize=7, color="grey")
fig.subplots_adjust(right=0.86, bottom=0.18)
out = FIG / f"band_share_cat{CATEGORY}_{day:%Y%m%d}.png"
fig.savefig(out, dpi=200, facecolor="white")
plt.show()
print("Saved", out)